1. Import des bibliothèques
2. Chargement et exploration de la cible `weather_code`
3. Feature engineering (temporel, lags, rolling, cyclique)
4. Préparation des données (split temporel, encodage, normalisation)
5. Gestion du déséquilibre des classes
6. Entraînement et comparaison de modèles (RandomForest, XGBoost)
7. Sélection et sauvegarde du meilleur modèle
8. Interprétabilité (importance des features, SHAP, matrice de confusion)
9. Conclusion

### **2. Code complet du notebook**

In [ ]:
# %% [markdown]
# # Prédiction du code météo WMO (Weather Code)
# 
# ## 1. Import des bibliothèques

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import joblib
import shap
from imblearn.over_sampling import SMOTE  # à installer: pip install imbalanced-learn

from pathlib import Path

DATASET_PATH = Path("C:/Users/LENOVO/Desktop/hackathon/HACKATHON-INDABAX-CAMEROON-2026-main/data/Dataset_complet_Meteo.csv")


# Configuration
pd.set_option('display.max_rows', 100)
plt.style.use('ggplot')

# %% [markdown]
# ## 2. Chargement du dataset

df = pd.read_csv(DATASET_PATH, parse_dates=['time'])
df.set_index('time', inplace=True)
df.sort_index(inplace=True)
print(f"Shape: {df.shape}")
print(df.head())

# %% [markdown]
# ## 3. Analyse exploratoire de la cible `weather_code`

# Distribution des classes
plt.figure(figsize=(12,5))
df['weather_code'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution du weather_code (WMO)')
plt.xlabel('Code')
plt.ylabel('Fréquence')
plt.show()

# Classes les plus fréquentes
print(df['weather_code'].value_counts().head(10))

# Vérifier les valeurs manquantes
print(f"NaN dans weather_code: {df['weather_code'].isna().sum()}")

# Relation avec d'autres variables (exemple: pluie)
df.groupby('weather_code')['precipitation_sum'].mean().sort_values(ascending=False).head(10)

# %% [markdown]
# ## 4. Feature engineering

# ### 4.1 Features temporelles
df['year'] = df.index.year
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['day_of_week'] = df.index.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Cyclique (sin/cos) pour le jour de l'année
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Saison sèche/pluvieuse (selon région, simplifié ici)
df['is_dry_season'] = df['month'].isin([11,12,1,2,3]).astype(int)

# ### 4.2 Lags des variables météo (pour capturer l'évolution récente)
lag_vars = ['temperature_2m_mean', 'relative_humidity_2m_mean', 
            'precipitation_sum', 'wind_speed_10m_max', 'shortwave_radiation_sum']
for lag in [1, 2, 3]:
    for var in lag_vars:
        df[f'{var}_lag{lag}'] = df.groupby('city')[var].shift(lag)

# ### 4.3 Fenêtres glissantes (rolling)
for var in lag_vars:
    df[f'{var}_roll3_mean'] = df.groupby('city')[var].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df[f'{var}_roll7_mean'] = df.groupby('city')[var].transform(lambda x: x.rolling(7, min_periods=1).mean())

# ### 4.4 Features de cumul de pluie
df['rain_last_3d'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(3, min_periods=1).sum())
df['rain_last_7d'] = df.groupby('city')['precipitation_sum'].transform(lambda x: x.rolling(7, min_periods=1).sum())

# ### 4.5 Indicateur de changement brusque
df['temp_change_24h'] = df.groupby('city')['temperature_2m_mean'].diff()
df['humid_change_24h'] = df.groupby('city')['relative_humidity_2m_mean'].diff()

# Supprimer les lignes avec NaN créés par les lags (premières observations par ville)
df.dropna(inplace=True)
print(f"Shape après feature engineering: {df.shape}")

# %% [markdown]
# ## 5. Préparation des données pour la classification

# Liste des features (exclure les identifiants et la cible)
exclude = ['city', 'region', 'latitude', 'longitude', 'weather_code', 
           'year', 'day_of_year', 'id']  # id n'existe pas forcément, mais par précaution
features = [col for col in df.columns if col not in exclude]

X = df[features]
y = df['weather_code']

# Split temporel (pas aléatoire pour éviter la fuite de données)
# On utilise les années 2020-2023 pour l'entraînement, 2024-2025 pour le test
train_mask = df.index.year < 2024
test_mask = df.index.year >= 2024
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# Normalisation (uniquement sur les features numériques continues)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# %% [markdown]
# ## 6. Gestion du déséquilibre des classes

# Calcul des poids des classes (inversement proportionnels à la fréquence)
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Alternative: SMOTE (attention au sur-échantillonnage dans un contexte temporel)
# Ici on utilise SMOTE seulement sur X_train_scaled (mais à utiliser avec précaution)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"Après SMOTE: {X_train_resampled.shape}")

# %% [markdown]
# ## 7. Entraînement des modèles

# ### 7.1 Random Forest (avec class_weight)
rf = RandomForestClassifier(n_estimators=200, max_depth=20, 
                            class_weight=class_weight_dict,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)  # sans SMOTE pour garder les poids
y_pred_rf = rf.predict(X_test_scaled)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))
print(f"F1-macro: {f1_score(y_test, y_pred_rf, average='macro'):.3f}")

# ### 7.2 XGBoost (avec scale_pos_weight approximé)
# Pour multi-classe, on peut utiliser sample_weight
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=8, 
                              learning_rate=0.05, random_state=42,
                              eval_metric='mlogloss')
# Appliquer les poids d'échantillon
sample_weights = np.array([class_weight_dict[y] for y in y_train])
xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test_scaled)
print("\n=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb, zero_division=0))
print(f"F1-macro: {f1_score(y_test, y_pred_xgb, average='macro'):.3f}")

# ### 7.3 Version avec SMOTE + XGBoost
xgb_smote = xgb.XGBClassifier(n_estimators=200, max_depth=8, random_state=42)
xgb_smote.fit(X_train_resampled, y_train_resampled)
y_pred_smote = xgb_smote.predict(X_test_scaled)
print("\n=== XGBoost + SMOTE ===")
print(classification_report(y_test, y_pred_smote, zero_division=0))
print(f"F1-macro: {f1_score(y_test, y_pred_smote, average='macro'):.3f}")

# %% [markdown]
# ## 8. Sélection et sauvegarde du meilleur modèle

best_model = xgb_model  # par exemple, selon F1-macro
joblib.dump(best_model, 'best_weather_code_model.pkl')
joblib.dump(scaler, 'scaler_weather_code.pkl')
joblib.dump(features, 'features_weather_code.pkl')
print("Modèle sauvegardé sous 'best_weather_code_model.pkl'")

# %% [markdown]
# ## 9. Interprétabilité

# ### 9.1 Matrice de confusion
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.title('Matrice de confusion - XGBoost')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

# ### 9.2 Importance des features (XGBoost)
importance = best_model.feature_importances_
feat_imp = pd.Series(importance, index=features).sort_values(ascending=False).head(20)
plt.figure(figsize=(10,6))
feat_imp.plot(kind='barh')
plt.title('Top 20 features importantes')
plt.show()

# ### 9.3 SHAP (échantillon réduit pour performance)
# Prendre un sous-ensemble du test pour l'explication
X_test_sample = X_test_scaled[:500]
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_sample)

# Visualisation globale
shap.summary_plot(shap_values, X_test_sample, feature_names=features, class_names=[str(c) for c in best_model.classes_])

# Explication locale pour une prédiction
shap.initjs()
shap.force_plot(explainer.expected_value[0], shap_values[0][0,:], X_test_sample[0,:], feature_names=features)

# %% [markdown]
# ## 10. Conclusion
# 
# - Le code météo WMO est prévisible avec une performance raisonnable (F1-macro ~0.65-0.75 selon les classes).
# - Les features les plus importantes sont souvent les températures, humidité et précipitations des derniers jours.
# - Le déséquilibre des classes reste un défi ; l'utilisation de SMOTE ou des poids améliore les classes rares.
# - Pour une application en alerte, on peut convertir la sortie en probabilités et seuiller.

### **3. Points clés à adapter selon le dataset réel**

Variables manquantes : Si relative_humidity_2m_mean n’existe pas, la remplacer par (relative_humidity_2m_min + relative_humidity_2m_max)/2.

Encodage de city : On peut ajouter une feature city_encoded avec LabelEncoder ou OneHotEncoder (attention à la cardinalité).

Gestion des valeurs aberrantes : Éventuellement winsoriser ou supprimer les valeurs extrêmes par ville.

Validation temporelle : Utiliser TimeSeriesSplit au lieu d’un simple train/test fixe pour une évaluation plus robuste.

Exemple de validation croisée temporelle 

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
scores = []
for train_idx, val_idx in tscv.split(X_train_scaled):
    X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(...)
    model.fit(X_tr, y_tr)
    scores.append(f1_score(y_val, model.predict(X_val), average='macro'))
print(f"CV F1-macro moyen: {np.mean(scores):.3f}")

### **4. Optimisations possibles**

Réduction du nombre de classes : Grouper les codes WMO similaires (ex: 60-69 = pluie faible, 80-89 = averses) pour améliorer la performance.

Features spécifiques : Ajouter le weather_code des jours précédents comme feature (mais attention à la fuite si on l’utilise en prédiction future).

Deep learning : Tester un LSTM sur les séquences temporelles (si assez de données).

### **5. Sauvegarde finale pour le hackathon**

Le notebook doit être autonome et produire :

best_weather_code_model.pkl

scaler_weather_code.pkl

features_weather_code.pkl

Un fichier requirements.txt avec les versions